In [1]:
# Tdownload Perseus (Grec classique / archaïque)
!git clone https://github.com/UniversalDependencies/UD_Ancient_Greek-Perseus.git /kaggle/working/UD_Perseus

# Download PROIEL (Nouveau Testament / Hérodote)
!git clone https://github.com/UniversalDependencies/UD_Ancient_Greek-PROIEL.git /kaggle/working/UD_PROIEL

print("Download finished")

fatal: destination path '/kaggle/working/UD_Perseus' already exists and is not an empty directory.
fatal: destination path '/kaggle/working/UD_PROIEL' already exists and is not an empty directory.
Download finished


In [ ]:
# 1. Install the specialized library for parsing CoNLL-U format files
!pip install conllu

from conllu import parse_incr

# 2. Define the path to the Perseus test corpus file
file_path = "/kaggle/working/UD_Perseus/grc_perseus-ud-test.conllu"

print("Opening the evaluation dataset...\n")

try:
    with open(file_path, "r", encoding="utf-8") as file:
        # 3. Extract and process the first sentence from the corpus
        for sentence in parse_incr(file):
            print(f" Original Sentence: {sentence.metadata['text']}")
            print("-" * 70)
            print(f"{'TOKEN':<15} | {'POS TAG':<15} | {'MORPHOLOGICAL FEATURES'}")
            print("-" * 70)
            
            # 4. Iterate through each token in the sentence
            for token in sentence:
                # 'form': The exact token/word in the text
                # 'upos': Universal Part-of-Speech tag (e.g., Verb, Noun)
                # 'feats': Morphological features (case, gender, number, tense, etc.)
                token_text = str(token['form'])
                pos_tag = str(token['upos'])
                
                # Format the morphological features for better readability
                morph_features = token['feats']
                if morph_features:
                    morph_str = ", ".join([f"{k}={v}" for k, v in morph_features.items()])
                else:
                    morph_str = "None"
                
                print(f"{token_text:<15} | {pos_tag:<15} | {morph_str}")
            
            # Break after the first sentence to limit output
            break
            
except FileNotFoundError:
    print(f" Error: The file {file_path} was not found.")
    print("Please ensure the dataset was successfully downloaded in the previous cell.")

In [ ]:
# 1. Mettre à jour et installer les bibliothèques
!pip install --upgrade pip setuptools wheel -q
!pip install -U conllu transformers datasets evaluate seqeval accelerate -q

import os

# Téléchargement direct via Git 
chemin_modele_local = "/kaggle/working/Ancient-Greek-BERT"

if not os.path.exists(chemin_modele_local):
    print("Telechargement du modele via Git (Methode anti-blocage)...")
    !git lfs install
    !git clone https://huggingface.co/pranaydeeps/Ancient-Greek-BERT /kaggle/working/Ancient-Greek-BERT
else:
    print("Le modele est deja telecharge sur le disque Kaggle.")
# ---------------------------------------------------------------

import torch
from conllu import parse_incr
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification
)
import evaluate
import numpy as np

# 2. Définir les étiquettes POS standard des Universal Dependencies
POS_TAGS = [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM", 
    "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X", "_"
]
tag2id = {tag: id for id, tag in enumerate(POS_TAGS)}
id2tag = {id: tag for id, tag in enumerate(POS_TAGS)}

# 3. Fonction pour parser les fichiers CoNLL-U en Datasets Hugging Face
def load_ud_data(file_path):
    data = {"tokens": [], "pos_tags": []}
    with open(file_path, "r", encoding="utf-8") as f:
        for sentence in parse_incr(f):
            tokens = []
            tags = []
            for token in sentence:
                if isinstance(token['id'], int): 
                    tokens.append(str(token['form']))
                    tag = str(token['upos']) if token['upos'] else "_"
                    tags.append(tag2id.get(tag, tag2id["_"]))
            data["tokens"].append(tokens)
            data["pos_tags"].append(tags)
    return Dataset.from_dict(data)

print("Chargement des jeux de donnees Perseus...")
train_dataset = load_ud_data("/kaggle/working/UD_Perseus/grc_perseus-ud-train.conllu")
val_dataset = load_ud_data("/kaggle/working/UD_Perseus/grc_perseus-ud-dev.conllu")
test_dataset = load_ud_data("/kaggle/working/UD_Perseus/grc_perseus-ud-test.conllu")

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# 4. Initialiser le Tokenizer et le modèle à partir du DOSSIER LOCAL
print(f"Chargement du tokenizer et du modele depuis le dossier : {chemin_modele_local}")

# Au lieu de télécharger, il lit directement les fichiers qu'on vient de cloner avec Git
tokenizer = AutoTokenizer.from_pretrained(chemin_modele_local, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(
    chemin_modele_local, 
    num_labels=len(POS_TAGS),
    id2label=id2tag,
    label2id=tag2id,
    local_files_only=True
)

# 5. Tokenisation et alignement des étiquettes
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=256
    )
    
    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenisation et alignement des etiquettes...")
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

# 6. Configuration des métriques d'évaluation
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [POS_TAGS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [POS_TAGS[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 7. Configuration de l'entraînement
print("Configuration de l'entraineur (Trainer)...")
training_args = TrainingArguments(
    output_dir="/kaggle/working/baseline-pos-perseus",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer, # CORRECTION ICI : processing_class au lieu de tokenizer
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 8. Lancer l'entraînement et l'évaluation
print("\nDemarrage du Fine-tuning de la Baseline...")
trainer.train()

print("\nEvaluation sur l'ensemble de test...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print("-" * 50)
print(f"Precision (Accuracy) de la Baseline : {test_results['eval_accuracy']:.4f}")
print(f"Score F1 de la Baseline : {test_results['eval_f1']:.4f}")
print("-" * 50)

In [ ]:
# 1. Mettre à jour et installer les bibliothèques
!pip install --upgrade pip setuptools wheel -q
!pip install -U conllu transformers datasets evaluate seqeval accelerate -q

import os

# --- LA SOLUTION INFAILLIBLE : Téléchargement direct via Git ---
# On contourne le téléchargeur Python de Hugging Face qui plante
chemin_modele_local = "/kaggle/working/Ancient-Greek-BERT"

if not os.path.exists(chemin_modele_local):
    print("Telechargement du modele via Git (Methode anti-blocage)...")
    !git lfs install
    !git clone https://huggingface.co/pranaydeeps/Ancient-Greek-BERT /kaggle/working/Ancient-Greek-BERT
else:
    print("Le modele est deja telecharge sur le disque Kaggle.")
# ---------------------------------------------------------------

import torch
from conllu import parse_incr
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification
)
import evaluate
import numpy as np

# 2. Définir les étiquettes POS standard des Universal Dependencies
POS_TAGS = [
    "ADJ", "ADP", "ADV", "AUX", "CCONJ", "DET", "INTJ", "NOUN", "NUM", 
    "PART", "PRON", "PROPN", "PUNCT", "SCONJ", "SYM", "VERB", "X", "_"
]
tag2id = {tag: id for id, tag in enumerate(POS_TAGS)}
id2tag = {id: tag for id, tag in enumerate(POS_TAGS)}

# 3. Fonction pour parser les fichiers CoNLL-U en Datasets Hugging Face
def load_ud_data(file_path):
    data = {"tokens": [], "pos_tags": []}
    with open(file_path, "r", encoding="utf-8") as f:
        for sentence in parse_incr(f):
            tokens = []
            tags = []
            for token in sentence:
                if isinstance(token['id'], int): 
                    tokens.append(str(token['form']))
                    tag = str(token['upos']) if token['upos'] else "_"
                    tags.append(tag2id.get(tag, tag2id["_"]))
            data["tokens"].append(tokens)
            data["pos_tags"].append(tags)
    return Dataset.from_dict(data)

print("Chargement des jeux de donnees PROIEL...")
train_dataset = load_ud_data("/kaggle/working/UD_PROIEL/grc_proiel-ud-train.conllu")
val_dataset = load_ud_data("/kaggle/working/UD_PROIEL/grc_proiel-ud-dev.conllu")
test_dataset = load_ud_data("/kaggle/working/UD_PROIEL/grc_proiel-ud-test.conllu")

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# 4. Initialiser le Tokenizer et le modèle à partir du DOSSIER LOCAL
print(f"Chargement du tokenizer et du modele depuis le dossier : {chemin_modele_local}")

# Au lieu de télécharger, il lit directement les fichiers qu'on vient de cloner avec Git
tokenizer = AutoTokenizer.from_pretrained(chemin_modele_local, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(
    chemin_modele_local, 
    num_labels=len(POS_TAGS),
    id2label=id2tag,
    label2id=tag2id,
    local_files_only=True
)

# 5. Tokenisation et alignement des étiquettes
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=256
    )
    
    labels = []
    for i, label in enumerate(examples["pos_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenisation et alignement des etiquettes...")
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

# 6. Configuration des métriques d'évaluation
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [POS_TAGS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [POS_TAGS[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 7. Configuration de l'entraînement
print("Configuration de l'entraineur (Trainer)...")
training_args = TrainingArguments(
    output_dir="/kaggle/working/baseline-pos-proiel",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer, 
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 8. Lancer l'entraînement et l'évaluation
print("\nDemarrage du Fine-tuning de la Baseline sur PROIEL...")
trainer.train()

print("\nEvaluation sur l'ensemble de test PROIEL...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print("-" * 50)
print(f"Precision (Accuracy) de la Baseline PROIEL : {test_results['eval_accuracy']:.4f}")
print(f"Score F1 de la Baseline PROIEL : {test_results['eval_f1']:.4f}")
print("-" * 50)

In [ ]:
!rm -rf ~/.cache/huggingface/

In [ ]:
# 1. Mettre a jour et installer les bibliotheques
!pip install --upgrade pip setuptools wheel -q
!pip install -U conllu transformers datasets evaluate seqeval accelerate -q

import os
import torch
import numpy as np
from conllu import parse_incr
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification
)
import evaluate

# 2. Verifier le modele local
chemin_modele_local = "/kaggle/working/Ancient-Greek-BERT"
if not os.path.exists(chemin_modele_local):
    print("Telechargement du modele via Git...")
    !git lfs install
    !git clone https://huggingface.co/pranaydeeps/Ancient-Greek-BERT /kaggle/working/Ancient-Greek-BERT
else:
    print("Le modele est deja telecharge sur le disque Kaggle.")

# 3. Fonction utilitaire pour extraire les traits morphologiques en chaine de caracteres
def get_morph_string(feats):
    if not feats:
        return "_"
    # Trier les cles pour garantir que "Case=Nom|Gender=Masc" soit identique a "Gender=Masc|Case=Nom"
    return "|".join([f"{k}={v}" for k, v in sorted(feats.items())])

# 4. Premier passage : Collecter toutes les combinaisons morphologiques possibles
print("Analyse des fichiers pour identifier toutes les combinaisons morphologiques...")
fichiers_perseus = [
    "/kaggle/working/UD_Perseus/grc_perseus-ud-train.conllu",
    "/kaggle/working/UD_Perseus/grc_perseus-ud-dev.conllu",
    "/kaggle/working/UD_Perseus/grc_perseus-ud-test.conllu"
]

tags_uniques = set(["_"])
for fichier in fichiers_perseus:
    with open(fichier, "r", encoding="utf-8") as f:
        for sentence in parse_incr(f):
            for token in sentence:
                if isinstance(token['id'], int):
                    tags_uniques.add(get_morph_string(token['feats']))

MORPH_TAGS = sorted(list(tags_uniques))
tag2id = {tag: id for id, tag in enumerate(MORPH_TAGS)}
id2tag = {id: tag for id, tag in enumerate(MORPH_TAGS)}
print(f"Nombre de classes morphologiques uniques trouvees : {len(MORPH_TAGS)}")

# 5. Fonction pour parser les fichiers avec les traits morphologiques
def load_morph_data(file_path):
    data = {"tokens": [], "morph_tags": []}
    with open(file_path, "r", encoding="utf-8") as f:
        for sentence in parse_incr(f):
            tokens = []
            tags = []
            for token in sentence:
                if isinstance(token['id'], int): 
                    tokens.append(str(token['form']))
                    tag = get_morph_string(token['feats'])
                    tags.append(tag2id.get(tag, tag2id["_"]))
            data["tokens"].append(tokens)
            data["morph_tags"].append(tags)
    return Dataset.from_dict(data)

print("Chargement des donnees Perseus pour la morphologie...")
train_dataset = load_morph_data(fichiers_perseus[0])
val_dataset = load_morph_data(fichiers_perseus[1])
test_dataset = load_morph_data(fichiers_perseus[2])

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# 6. Initialiser le Tokenizer et le modele a partir du dossier local
print("Chargement du tokenizer et du modele...")
tokenizer = AutoTokenizer.from_pretrained(chemin_modele_local, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(
    chemin_modele_local, 
    num_labels=len(MORPH_TAGS),
    id2label=id2tag,
    label2id=tag2id,
    local_files_only=True
)

# 7. Tokenisation et alignement
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=256
    )
    
    labels = []
    for i, label in enumerate(examples["morph_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenisation et alignement des etiquettes...")
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

# 8. Configuration des metriques
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [MORPH_TAGS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [MORPH_TAGS[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 9. Configuration de l'entraineur
print("Configuration de l'entraineur...")
training_args = TrainingArguments(
    output_dir="/kaggle/working/baseline-morph-perseus",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 10. Lancement
print("\nDemarrage de l'entrainement pour la morphologie...")
trainer.train()

print("\nEvaluation sur l'ensemble de test...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print("-" * 50)
print(f"Precision (Accuracy) Morphologie : {test_results['eval_accuracy']:.4f}")
print(f"Score F1 Morphologie : {test_results['eval_f1']:.4f}")
print("-" * 50)

In [ ]:
# 1. Mettre a jour et installer les bibliotheques
!pip install --upgrade pip setuptools wheel -q
!pip install -U transformers datasets evaluate seqeval accelerate -q

import os
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification
)
import evaluate

# 2. Verifier le modele local
chemin_modele_local = "/kaggle/working/Ancient-Greek-BERT"
if not os.path.exists(chemin_modele_local):
    print("Telechargement du modele via Git...")
    !git lfs install
    !git clone https://huggingface.co/pranaydeeps/Ancient-Greek-BERT /kaggle/working/Ancient-Greek-BERT
else:
    print("Le modele est deja telecharge sur le disque Kaggle.")

# 3. Generation d'un jeu de donnees d'exemple (Dummy Data) pour valider le pipeline
# Format CoNLL standard : "Mot Etiquette"
print("Creation d'un mini-jeu de donnees NER d'exemple pour tester le pipeline...")
dossier_ner = "/kaggle/working/NER_Data"
os.makedirs(dossier_ner, exist_ok=True)

donnees_exemple = """Σωκράτης B-PER
ἐν O
Ἀθήναις B-LOC
ζῇ O
. O

Οἱ O
Σπαρτιᾶται B-ORG
μάχονται O
. O

Πλάτων B-PER
λέγει O
τὸν O
λόγον O
. O
"""

# On cree des fichiers factices pour train, dev et test
for split in ["train.txt", "dev.txt", "test.txt"]:
    with open(os.path.join(dossier_ner, split), "w", encoding="utf-8") as f:
        # On multiplie les exemples pour avoir assez de donnees pour que l'entrainement se lance
        f.write((donnees_exemple + "\n") * 20)

# 4. Fonction pour lire le format CoNLL-2003 NER
def lire_donnees_ner(chemin_fichier):
    donnees = {"tokens": [], "ner_tags": []}
    mots = []
    tags = []
    
    with open(chemin_fichier, "r", encoding="utf-8") as f:
        for ligne in f:
            ligne = ligne.strip()
            if not ligne: # Ligne vide = fin de la phrase
                if mots:
                    donnees["tokens"].append(mots)
                    donnees["ner_tags"].append(tags)
                    mots = []
                    tags = []
            else:
                parties = ligne.split()
                if len(parties) >= 2:
                    mots.append(parties[0])
                    tags.append(parties[-1])
                    
    # Ajouter la derniere phrase si le fichier ne finit pas par une ligne vide
    if mots:
        donnees["tokens"].append(mots)
        donnees["ner_tags"].append(tags)
        
    return Dataset.from_dict(donnees)

print("Chargement des donnees NER...")
train_dataset = lire_donnees_ner(os.path.join(dossier_ner, "train.txt"))
val_dataset = lire_donnees_ner(os.path.join(dossier_ner, "dev.txt"))
test_dataset = lire_donnees_ner(os.path.join(dossier_ner, "test.txt"))

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# 5. Definition des etiquettes NER
NER_TAGS = ["O", "B-PER", "I-PER", "B-LOC", "I-LOC", "B-ORG", "I-ORG", "B-MISC", "I-MISC"]
tag2id = {tag: id for id, tag in enumerate(NER_TAGS)}
id2tag = {id: tag for id, tag in enumerate(NER_TAGS)}

# Remplacer les etiquettes texte par leurs IDs dans les datasets
def encoder_tags(exemple):
    exemple["ner_tags"] = [tag2id[tag] for tag in exemple["ner_tags"]]
    return exemple

datasets = datasets.map(encoder_tags)

# 6. Initialiser le Tokenizer et le modele a partir du dossier local
print("Chargement du tokenizer et du modele...")
tokenizer = AutoTokenizer.from_pretrained(chemin_modele_local, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(
    chemin_modele_local, 
    num_labels=len(NER_TAGS),
    id2label=id2tag,
    label2id=tag2id,
    local_files_only=True,
    ignore_mismatched_sizes=True # Indispensable car le nombre d'etiquettes change par rapport a la grammaire
)

# 7. Tokenisation et alignement
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=256
    )
    
    labels = []
    for i, label_list in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label_list[word_idx])
            else:
                # Pour le NER, on met a -100 les sous-mots suivants
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenisation et alignement des etiquettes NER...")
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

# 8. Configuration des metriques
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [NER_TAGS[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [NER_TAGS[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

# 9. Configuration de l'entraineur
print("Configuration de l'entraineur pour le NER...")
training_args = TrainingArguments(
    output_dir="/kaggle/working/baseline-ner",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# 10. Lancement
print("\nDemarrage de l'entrainement pour la Reconnaissance d'Entites Nommees (NER)...")
trainer.train()

print("\nEvaluation sur l'ensemble de test NER...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print("-" * 50)
print(f"Precision (Accuracy) NER : {test_results['eval_accuracy']:.4f}")
print(f"Score F1 NER : {test_results['eval_f1']:.4f}")
print("-" * 50)

In [2]:
# 1. Mettre a jour et installer les bibliotheques
!pip install --upgrade pip setuptools wheel -q
!pip install -U conllu transformers datasets evaluate seqeval accelerate -q

import os
import torch
import numpy as np
from conllu import parse_incr
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    TrainingArguments, 
    Trainer, 
    DataCollatorForTokenClassification
)
import evaluate

# 2. Verifier le modele local
chemin_modele_local = "/kaggle/working/Ancient-Greek-BERT"
if not os.path.exists(chemin_modele_local):
    print("Telechargement du modele via Git...")
    !git lfs install
    !git clone https://huggingface.co/pranaydeeps/Ancient-Greek-BERT /kaggle/working/Ancient-Greek-BERT
else:
    print("Le modele est deja telecharge sur le disque Kaggle.")

# 3. Premier passage : Collecter tous les lemmes uniques du dictionnaire (formes de base)
print("Analyse des fichiers pour identifier tous les lemmes uniques...")
fichiers_perseus = [
    "/kaggle/working/UD_Perseus/grc_perseus-ud-train.conllu",
    "/kaggle/working/UD_Perseus/grc_perseus-ud-dev.conllu",
    "/kaggle/working/UD_Perseus/grc_perseus-ud-test.conllu"
]

lemmes_uniques = set(["_"]) # Le tiret bas sert de valeur par defaut pour les inconnus
for fichier in fichiers_perseus:
    with open(fichier, "r", encoding="utf-8") as f:
        for sentence in parse_incr(f):
            for token in sentence:
                if isinstance(token['id'], int):
                    lemme = str(token['lemma']) if token['lemma'] else "_"
                    lemmes_uniques.add(lemme)

LEMMAS = sorted(list(lemmes_uniques))
tag2id = {tag: id for id, tag in enumerate(LEMMAS)}
id2tag = {id: tag for id, tag in enumerate(LEMMAS)}
print(f"Nombre de lemmes uniques trouves (taille du dictionnaire) : {len(LEMMAS)}")

# 4. Fonction pour parser les fichiers et extraire les lemmes
def load_lemma_data(file_path):
    data = {"tokens": [], "lemma_tags": []}
    with open(file_path, "r", encoding="utf-8") as f:
        for sentence in parse_incr(f):
            tokens = []
            tags = []
            for token in sentence:
                if isinstance(token['id'], int): 
                    tokens.append(str(token['form']))
                    lemme = str(token['lemma']) if token['lemma'] else "_"
                    tags.append(tag2id.get(lemme, tag2id["_"]))
            data["tokens"].append(tokens)
            data["lemma_tags"].append(tags)
    return Dataset.from_dict(data)

print("Chargement des donnees Perseus pour la lemmatisation...")
train_dataset = load_lemma_data(fichiers_perseus[0])
val_dataset = load_lemma_data(fichiers_perseus[1])
test_dataset = load_lemma_data(fichiers_perseus[2])

datasets = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
    "test": test_dataset
})

# 5. Initialiser le Tokenizer et le modele a partir du dossier local
print("Chargement du tokenizer et du modele...")
tokenizer = AutoTokenizer.from_pretrained(chemin_modele_local, local_files_only=True)
model = AutoModelForTokenClassification.from_pretrained(
    chemin_modele_local, 
    num_labels=len(LEMMAS),
    id2label=id2tag,
    label2id=tag2id,
    local_files_only=True,
    ignore_mismatched_sizes=True # Indispensable car on a des milliers de lemmes (classes)
)

# 6. Tokenisation et alignement
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples["tokens"], truncation=True, is_split_into_words=True, max_length=256
    )
    
    labels = []
    for i, label in enumerate(examples["lemma_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                label_ids.append(label[word_idx])
            else:
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
        
    tokenized_inputs["labels"] = labels
    return tokenized_inputs

print("Tokenisation et alignement des lemmes...")
tokenized_datasets = datasets.map(tokenize_and_align_labels, batched=True)

# 7. Configuration des metriques (Exact Match Accuracy)
metric = evaluate.load("accuracy")

# OPTIMISATION MEMOIRE : On extrait la reponse (argmax) directement sur la carte graphique 
# lot par lot pour ne pas accumuler des gigaoctets de probabilites inutiles.
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    return logits.argmax(dim=-1)

def compute_metrics(p):
    predictions, labels = p
    # L'argmax a deja ete fait grace a preprocess_logits_for_metrics, plus besoin de le refaire ici.
    
    # Aplatir les listes et ignorer les labels -100 (padding/sous-mots)
    true_predictions = [
        p for prediction, label in zip(predictions, labels) for (p, l) in zip(prediction, label) if l != -100
    ]
    true_labels = [
        l for prediction, label in zip(predictions, labels) for (p, l) in zip(prediction, label) if l != -100
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "accuracy": results["accuracy"]
    }

# 8. Configuration de l'entraineur (Optimisee pour la memoire)
print("Configuration de l'entraineur...")
training_args = TrainingArguments(
    output_dir="/kaggle/working/baseline-lemmatisation-perseus",
    learning_rate=2e-5,
    per_device_train_batch_size=8, # Reduit a 8 pour eviter l'erreur CUDA
    per_device_eval_batch_size=8,  # Reduit a 8 pour eviter l'erreur CUDA
    num_train_epochs=3,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=50,
    report_to="none"
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics, # Activation de l'optimisation
)

# 9. Lancement
print("\nDemarrage de l'entrainement pour la Lemmatisation...")
trainer.train()

print("\nEvaluation sur l'ensemble de test...")
test_results = trainer.evaluate(tokenized_datasets["test"])
print("-" * 50)
print(f"Precision (Accuracy) Lemmatisation : {test_results['eval_accuracy']:.4f}")
print("-" * 50)

Le modele est deja telecharge sur le disque Kaggle.
Analyse des fichiers pour identifier tous les lemmes uniques...
Nombre de lemmes uniques trouves (taille du dictionnaire) : 13601
Chargement des donnees Perseus pour la lemmatisation...
Chargement du tokenizer et du modele...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: /kaggle/working/Ancient-Greek-BERT
Key                 | Status     | 
--------------------+------------+-
pooler.dense.bias   | UNEXPECTED | 
pooler.dense.weight | UNEXPECTED | 
classifier.bias     | MISSING    | 
classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenisation et alignement des lemmes...


Map:   0%|          | 0/11476 [00:00<?, ? examples/s]

Map:   0%|          | 0/1137 [00:00<?, ? examples/s]

... (9 outputs left out)


Evaluation sur l'ensemble de test...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Accuracy
4.844936,6.685789,3,0.639582


--------------------------------------------------
Precision (Accuracy) Lemmatisation : 0.6396
--------------------------------------------------


In [3]:
# 1. Installer les bibliotheques necessaires
!pip install --upgrade pip setuptools wheel -q
!pip install -U transformers datasets torch accelerate -q

import os
import torch
import torch.nn.functional as F
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModel, TrainingArguments, Trainer
from datasets import Dataset

# 2. Verifier le modele local
chemin_modele_local = "/kaggle/working/Ancient-Greek-BERT"
if not os.path.exists(chemin_modele_local):
    print("Telechargement du modele via Git...")
    !git lfs install
    !git clone https://huggingface.co/pranaydeeps/Ancient-Greek-BERT /kaggle/working/Ancient-Greek-BERT

# =====================================================================
# TACHE 1 : CLASSIFICATION DE DOCUMENTS (Attribution d'auteur)
# =====================================================================
print("\n" + "="*50)
print("TACHE 1 : PROTOTYPE DE CLASSIFICATION D'AUTEUR")
print("="*50)

# Mini-corpus d'entrainement (Citations d'Homer, Platon, et Nouveau Testament)
textes_train = [
    "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος", # Homere (Iliade)
    "ἄνδρα μοι ἔννεπε, μοῦσα, πολύτροπον", # Homere (Odyssee)
    "Σωκράτης δοκεῖ μοι ἄριστος εἶναι",   # Platon (Philosophie)
    "τὸν μὲν οὖν λόγον, ὦ ἄνδρες Ἀθηναῖοι", # Platon / Orateurs attiques
    "Ἐν ἀρχῇ ἦν ὁ λόγος", # Nouveau Testament
    "καὶ ὁ λόγος ἦν πρὸς τὸν θεόν" # Nouveau Testament
]
# 0 = Homere, 1 = Platon/Attique, 2 = Nouveau Testament
labels_train = [0, 0, 1, 1, 2, 2] 
noms_auteurs = ["Homère (Épique)", "Platon (Attique)", "Nouveau Testament (Koinè)"]

dataset = Dataset.from_dict({"text": textes_train, "label": labels_train})

print("Chargement du tokenizer et du modele de classification...")
tokenizer = AutoTokenizer.from_pretrained(chemin_modele_local, local_files_only=True)
# On charge le modele pour de la CLASSIFICATION DE SEQUENCE (pas de Token)
model_clf = AutoModelForSequenceClassification.from_pretrained(
    chemin_modele_local, 
    num_labels=3, 
    local_files_only=True,
    ignore_mismatched_sizes=True
)

def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=32)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# Un entrainement eclair (juste pour le prototype)
training_args = TrainingArguments(
    output_dir="/kaggle/working/prototype-clf",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    logging_steps=2,
    report_to="none"
)

trainer = Trainer(
    model=model_clf,
    args=training_args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
)

print("Entrainement rapide du classifieur...")
trainer.train()

# Test de classification
texte_mystere = "θεὰ ἄειδε τὴν ὀργὴν" # "Deesse chante la colere" (Style Homere)
inputs = tokenizer(texte_mystere, return_tensors="pt").to(model_clf.device)
with torch.no_grad():
    logits = model_clf(**inputs).logits
prediction = torch.argmax(logits, dim=-1).item()

print(f"\nTexte mystère : '{texte_mystere}'")
print(f"-> L'IA a classé ce texte comme : {noms_auteurs[prediction]}")


# =====================================================================
# TACHE 2 : MOTEUR DE RECHERCHE SEMANTIQUE (Retrieval)
# =====================================================================
print("\n" + "="*50)
print("TACHE 2 : PROTOTYPE DE MOTEUR DE RECHERCHE (EMBEDDINGS)")
print("="*50)

# Pour la recherche, on utilise le modele de base (sans couche de classification)
print("Chargement du modele de base pour extraire le sens (Embeddings)...")
model_base = AutoModel.from_pretrained(chemin_modele_local, local_files_only=True)
model_base.eval() # Mode evaluation

# Base de donnees de documents
documents = [
    "Οἱ Ἀθηναῖοι ἐμάχοντο ἐν τῇ θαλάσσῃ.", # Les Atheniens se battaient sur la mer.
    "Ὁ Σωκράτης ἔπινε τὸ φάρμακον.",       # Socrate a bu le poison.
    "Οἱ θεοὶ κατοικοῦσι τὸν Ὄλυμπον."      # Les dieux habitent l'Olympe.
]

# Fonction pour transformer une phrase en vecteur mathematique (Embedding)
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=32)
    with torch.no_grad():
        outputs = model_base(**inputs)
    # On prend la moyenne des vecteurs de tous les mots pour representer la phrase entiere
    return outputs.last_hidden_state.mean(dim=1)

print("Vectorisation de la base de donnees (création des Embeddings)...")
doc_embeddings = torch.cat([get_embedding(doc) for doc in documents])

# La requete de l'utilisateur (Recherche d'un concept, pas des mots exacts)
requete = "Ποῦ ζῶσι οἱ ἀθάνατοι;" # "Où vivent les immortels ?"
print(f"\nRequete utilisateur : '{requete}'")

req_embedding = get_embedding(requete)

# Calcul de la similarite Cosinus entre la requete et tous les documents
similarites = F.cosine_similarity(req_embedding, doc_embeddings)

# Trouver le document avec le score le plus eleve
meilleur_index = torch.argmax(similarites).item()
meilleur_score = similarites[meilleur_index].item()

print("\n--- RESULTATS DE LA RECHERCHE ---")
for i, doc in enumerate(documents):
    print(f"Document {i+1} | Score de similarité : {similarites[i].item():.4f} | Texte : {doc}")

print(f"\n=> Le document le plus pertinent est le n°{meilleur_index + 1} avec un score de {meilleur_score:.4f}.")
print(f"=> Texte trouvé : '{documents[meilleur_index]}'")
print("="*50)

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
cuda-python 12.9.4 requires cuda-bindings~=12.9.4, but you have cuda-bindings 13.3.1 which is incompatible.
torchaudio 2.10.0+cu128 requires torch==2.10.0, but you have torch 2.13.0 which is incompatible.
torchvision 0.25.0+cu128 requires torch==2.10.0, but you have torch 2.13.0 which is incompatible.
cudf-cu12 26.2.1 requires cuda-toolkit[nvcc,nvrtc]==12.*, but you have cuda-toolkit 13.0.3.0 which is incompatible.
libcuvs-cu12 26.2.0 requires cuda-toolkit[cublas,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.3.0 

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: /kaggle/working/Ancient-Greek-BERT
Key               | Status  | 
------------------+---------+-
classifier.bias   | MISSING | 
classifier.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/6 [00:00<?, ? examples/s]

Entrainement rapide du classifieur...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss
2,2.327062
4,1.871346
6,1.471758
8,1.181216
10,1.070482


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Texte mystère : 'θεὰ ἄειδε τὴν ὀργὴν'
-> L'IA a classé ce texte comme : Nouveau Testament (Koinè)

TACHE 2 : PROTOTYPE DE MOTEUR DE RECHERCHE (EMBEDDINGS)
Chargement du modele de base pour extraire le sens (Embeddings)...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Vectorisation de la base de donnees (création des Embeddings)...

Requete utilisateur : 'Ποῦ ζῶσι οἱ ἀθάνατοι;'

--- RESULTATS DE LA RECHERCHE ---
Document 1 | Score de similarité : 0.6041 | Texte : Οἱ Ἀθηναῖοι ἐμάχοντο ἐν τῇ θαλάσσῃ.
Document 2 | Score de similarité : 0.6001 | Texte : Ὁ Σωκράτης ἔπινε τὸ φάρμακον.
Document 3 | Score de similarité : 0.6904 | Texte : Οἱ θεοὶ κατοικοῦσι τὸν Ὄλυμπον.

=> Le document le plus pertinent est le n°3 avec un score de 0.6904.
=> Texte trouvé : 'Οἱ θεοὶ κατοικοῦσι τὸν Ὄλυμπον.'
